> **VERSÃO GABARITO** — todas as células preenchidas. Use esta para projetar; distribua a outra para a turma.

# Análise de dados com Python

**Pergunta da aula:** os dados de áudio do Spotify conseguem descrever o que é cada gênero musical?

Você já sabe estatística descritiva. O que é novo aqui é só a ferramenta.
Em 100 minutos vamos percorrer uma análise exploratória inteira em Python:
olhar o dado, uma variável, uma variável por grupo, duas variáveis.

**Dados:** 32.833 músicas do Spotify, com 6 gêneros e uma dúzia de
características de áudio medidas pelo algoritmo da plataforma.
Fonte: [TidyTuesday, 2020-01-21](https://github.com/rfordatascience/tidytuesday/tree/main/data/2020/2020-01-21).

---

### Como usar este notebook

Um notebook é uma sequência de **células**. Célula de texto (como esta) e
célula de código. Para rodar uma célula de código: clique nela e aperte
**Shift + Enter**.

Quando aparecer **SUA VEZ**, a célula é sua: ou está vazia e o código está
no texto acima, ou tem lacunas `____` para preencher.

Se algo quebrar, leia a **última linha** da mensagem de erro. É lá que
está a informação útil.

## 0 · Preparando o ambiente  `⏱️ 0–20 min`

Em R você carrega pacotes com `library()`. Em Python é `import`.
O `as pd` cria um apelido: em vez de escrever `pandas` toda vez, escreve `pd`.

Os três pacotes de hoje:

- **pandas** — tabelas. É o `data.frame` do Python.
- **seaborn** — gráficos estatísticos.
- **matplotlib** — o motor gráfico por baixo do seaborn.

A célula abaixo **instala** (se faltar) e **carrega** os três.
No Colab, rode **uma vez** no começo da aula — não precisa instalar nada à mão.

In [ ]:
# Instala o que faltar e carrega os pacotes da aula (Colab e Jupyter).
# Rode esta célula uma vez no começo.
%pip install -q pandas seaborn matplotlib

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

print("pandas", pd.__version__)
print("seaborn", sns.__version__)
print("Pronto — ambiente ok.")

Agora os dados. `read_csv` lê um arquivo CSV — e aceita uma URL
diretamente, sem download manual.

O resultado é um **DataFrame**, que guardamos na variável `df`.

In [ ]:
URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-01-21/spotify_songs.csv"

df = pd.read_csv(URL)

print("Carregado.")

---

## 1 · O que é esse dado?  `⏱️ 20–30 min`

Antes de qualquer análise: quantas linhas, quantas colunas, o que cada
uma significa.

### SUA VEZ — digite você mesmo

`df.head()` mostra as primeiras linhas. Digite na célula abaixo:

```python
df.head()
```

Repare: sem parênteses vazios o Python devolve outra coisa. Teste depois
se quiser ver a diferença.

In [ ]:
df.head()

### SUA VEZ — digite você mesmo

`df.shape` devolve (número de linhas, número de colunas). É o `dim()` do R.
Note que aqui **não** tem parênteses: `shape` é um atributo, não uma função.

```python
df.shape
```

In [ ]:
df.shape

`df.info()` lista as colunas, o tipo de cada uma e quantos valores
não-nulos existem. É o primeiro diagnóstico de qualquer tabela.

In [ ]:
df.info()

### O dicionário das variáveis

As que vamos usar hoje:

| coluna | o que é |
|---|---|
| `track_name`, `track_artist` | música e artista |
| `track_popularity` | popularidade de 0 a 100, medida pelo Spotify |
| `playlist_genre` | gênero: edm, latin, pop, r&b, rap, rock |
| `danceability` | 0 a 1 — o quanto a música serve para dançar |
| `energy` | 0 a 1 — intensidade e atividade |
| `valence` | 0 a 1 — "positividade" musical. Perto de 0 soa triste, perto de 1 soa alegre |
| `loudness` | volume médio em dB (valores negativos) |
| `acousticness` | 0 a 1 — confiança de que a música é acústica |
| `tempo` | batidas por minuto |
| `duration_ms` | duração em milissegundos |

Todas essas medidas de áudio são **estimativas de um algoritmo**, não
medidas físicas. Guarde isso para a discussão do fim.

### Uma armadilha: o que é uma linha?

Parece óbvio que cada linha é uma música. Vamos verificar contando
quantos identificadores únicos de música existem.

In [ ]:
n_linhas = len(df)
n_musicas = df["track_id"].nunique()

print("linhas na tabela:", n_linhas)
print("músicas distintas:", n_musicas)

Não bate. A mesma música aparece em mais de uma playlist, e cada
aparição virou uma linha. A unidade de observação não é *música*, é
**música-em-playlist**.

Isso não é um detalhe de programação, é uma decisão de análise: músicas
que estão em muitas playlists pesam mais em qualquer média que
calcularmos. Hoje vamos trabalhar com músicas distintas.

`drop_duplicates` fica com a primeira aparição de cada `track_id`.

In [ ]:
musicas = df.drop_duplicates(subset="track_id")
n_antes = len(df)
n_depois = len(musicas)

print("antes:", n_antes)
print("depois:", n_depois)

E os valores faltantes? `isna()` marca cada célula como faltante ou
não, e `.sum()` conta por coluna.

In [ ]:
musicas.isna().sum()

### SUA VEZ — troque a coluna

`value_counts()` conta quantas vezes cada valor aparece. É o `table()` do R.

Rode como está, depois troque `"playlist_genre"` por `"playlist_subgenre"`
e rode de novo.

In [ ]:
musicas["playlist_genre"].value_counts()

---

## 2 · Uma variável: como se distribui a popularidade?  `⏱️ 30–45 min`

Primeiro passo de qualquer EDA: olhar uma variável de cada vez.
`describe()` devolve o resumo de cinco números mais média e desvio.

In [ ]:
musicas["track_popularity"].describe()

Média e mediana próximas. Parece bem comportada.

Agora o histograma. No seaborn, um gráfico é uma linha: você diz de onde
vêm os dados (`data`) e qual coluna vai no eixo x.

In [ ]:
sns.histplot(data=musicas, x="track_popularity", bins=50)
plt.title("Distribuição da popularidade")
plt.show()

Aquela barra em zero não é ruído.

Para contar quantas músicas têm popularidade exatamente 0, usamos um
**filtro booleano**: a expressão `musicas["track_popularity"] == 0`
devolve uma coluna de Verdadeiro/Falso, e somar booleanos conta os
verdadeiros.

In [ ]:
zeros = (musicas["track_popularity"] == 0).sum()

proporcao = round(100 * zeros / len(musicas), 1)

print("músicas com popularidade 0:", zeros)
print("proporção:", proporcao, "%")

Quase uma em cada dez. A pergunta que isso levanta é estatística,
não computacional: **são zeros reais ou é ausência de informação
codificada como zero?** O dicionário não diz. Uma análise séria
investigaria antes de calcular qualquer média.

Note também o efeito dessa massa em zero sobre as medidas de centro.

In [ ]:
musicas["track_popularity"].agg(["mean", "median", "std"])

### SUA VEZ — troque a variável

Copie o histograma acima trocando `"track_popularity"` por `"tempo"`.
Depois teste com `"danceability"`.

Qual das três distribuições mais te surpreende?

In [ ]:
sns.histplot(data=musicas, x="tempo", bins=50)
plt.title("Distribuição do tempo (BPM)")
plt.show()

---

## 3 · Uma variável por grupo: qual gênero é o mais triste?  `⏱️ 45–65 min`

`valence` mede a positividade musical de 0 a 1. Vamos comparar os
gêneros.

O verbo é `groupby`: separa as linhas por gênero e aplica uma função a
cada grupo. É o `aggregate()` do R.

### SUA VEZ — preencha as lacunas

O código completo é:

```python
musicas.groupby("playlist_genre")["valence"].mean()
```

Leia da esquerda para a direita: *pegue as músicas, agrupe por gênero,
selecione a coluna valence, tire a média*.

Complete a célula abaixo.

In [ ]:
musicas.groupby("playlist_genre")["valence"].mean()

Ordenado fica mais legível. `sort_values()` ordena o resultado.

In [ ]:
musicas.groupby("playlist_genre")["valence"].mean().sort_values()

Média sozinha esconde coisa. `agg()` calcula várias medidas de uma
vez — passe os nomes numa lista.

In [ ]:
musicas.groupby("playlist_genre")["valence"].agg(["mean", "median", "std"]).sort_values("mean")

### SUA VEZ — preencha as lacunas

Tabela é bom, gráfico é melhor. O boxplot mostra a distribuição inteira
de cada grupo, não só o centro.

O código completo é:

```python
sns.boxplot(data=musicas, x="playlist_genre", y="valence")
plt.title("Positividade musical por gênero")
plt.show()
```

In [ ]:
sns.boxplot(data=musicas, x="playlist_genre", y="valence")
plt.title("Positividade musical por gênero")
plt.show()

**O achado.** EDM é o gênero com a menor positividade musical, e
Latin a maior. A diferença entre os dois é grande o bastante para
aparecer no boxplot inteiro, não só nas médias.

Guarde esse resultado — ele fica mais estranho na próxima célula.

### SUA VEZ — troque a variável

Troque `"valence"` por `"energy"` no boxplot e rode de novo.

In [ ]:
sns.boxplot(data=musicas, x="playlist_genre", y="energy")
plt.title("Energia por gênero")
plt.show()

O gênero de menor positividade é o de **maior** energia.

Isso diz algo sobre as duas medidas: energia e positividade não são a
mesma coisa. Música intensa não é música alegre. Uma faixa de EDM pode
ser alta, rápida e em tom menor ao mesmo tempo.

---

## 4 · Duas variáveis: o que anda junto?  `⏱️ 65–80 min`

Antes, um detalhe prático: criar uma coluna nova é uma atribuição
direta. Duração em milissegundos não se lê; em minutos, sim.

In [ ]:
musicas = musicas.copy()
musicas["duracao_min"] = musicas["duration_ms"] / 60000

musicas[["track_name", "duracao_min"]].head()

Agora o gráfico de dispersão entre energia e volume.

Com 28 mil pontos, a nuvem vira uma mancha sólida e você não enxerga
onde os dados se concentram. `alpha=0.1` deixa cada ponto quase
transparente: onde há muita sobreposição fica escuro, onde há poucos
pontos fica claro.

In [ ]:
sns.scatterplot(data=musicas, x="energy", y="loudness", alpha=0.1)
plt.title("Energia × volume")
plt.show()

Relação clara. Quantificando com `corr()`:

In [ ]:
musicas["energy"].corr(musicas["loudness"])

Para ver todos os pares de uma vez, `corr()` aplicado a várias
colunas devolve a matriz de correlações.

In [ ]:
variaveis = ["danceability", "energy", "valence", "loudness",
             "acousticness", "tempo", "duracao_min", "track_popularity"]

musicas[variaveis].corr().round(2)

Uma matriz de números é difícil de ler. O `heatmap` colore por
magnitude; `annot=True` mantém os números visíveis.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(musicas[variaveis].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0)
plt.title("Correlações entre as características de áudio")
plt.show()

### SUA VEZ — troque o par

Faça o gráfico de dispersão de `energy` contra `acousticness`, com
`alpha=0.1`. O heatmap já te disse mais ou menos o que esperar.

In [ ]:
sns.scatterplot(data=musicas, x="energy", y="acousticness", alpha=0.1)
plt.title("Energia × acústica")
plt.show()

---

## 5 · A pergunta que todo mundo faz  `⏱️ 80–85 min`

*O que faz uma música ser popular?*

A hipótese óbvia: música dançante bomba mais. Vamos olhar.

In [ ]:
sns.scatterplot(data=musicas, x="danceability", y="track_popularity", alpha=0.1)
plt.title("Dançabilidade × popularidade")
plt.show()

Uma nuvem sem forma. Para cada valor de dançabilidade existe música
de toda popularidade possível.

E não é só essa variável. A última linha do heatmap já tinha entregue,
mas vale olhar isolado — ordenado por magnitude.

In [ ]:
musicas[variaveis].corr()["track_popularity"].drop("track_popularity").sort_values()

Nenhuma característica de áudio passa de 0,15 em módulo. Sobre a
popularidade de uma música, essas doze medidas de áudio não dizem
praticamente nada.

Isso **não** é uma falha da análise. É o resultado. A popularidade de
uma música depende de coisas que não estão nesta tabela: divulgação,
quem é o artista, playlist editorial, TikTok, o ano em que saiu.

Uma análise exploratória que encontra ausência de relação fez o seu
trabalho. O erro seria procurar até achar alguma coisa.

---

## 6 · De volta à pergunta  `⏱️ 85–87 min`

> *Os dados de áudio conseguem descrever o que é cada gênero musical?*

**Descrevem gênero: sim.** Gêneros se separam de forma nítida em
positividade e em energia, e a separação aparece na distribuição
inteira, não só na média.

**Descrevem popularidade: não.** Todas as correlações com popularidade
são próximas de zero.

E o caminho até essas duas frases foi sempre o mesmo, independente da
linguagem: **olhar o dado → uma variável → uma variável por grupo →
duas variáveis.**

---

## 7 · Para onde isso leva  `⏱️ 87–100 min`

Hoje usamos duas bibliotecas. O ecossistema continua, e em geral a
tradução do R é direta:

| para | Python | equivalente em R |
|---|---|---|
| tabelas | **pandas** | `data.frame`, `dplyr` |
| gráficos | **seaborn**, **matplotlib** | `ggplot2` |
| álgebra e números | **NumPy** | vetores e matrizes do R base |
| modelos estatísticos | **statsmodels** | `lm()`, `glm()` |
| aprendizado de máquina | **scikit-learn** | `caret`, `tidymodels` |
| testes e distribuições | **SciPy** | `stats` do R base |

Com `statsmodels` você ajustaria uma regressão da popularidade nas
características de áudio — e, a julgar pelas correlações de hoje,
encontraria um R² baixíssimo. Que é exatamente o que a análise
exploratória já antecipou, e de graça.

Essa é a moral da aula: a EDA vem antes do modelo, e frequentemente
poupa você de ajustar um.

### Para brincar depois

Nada disso é para entregar. Se quiser continuar:

- Qual artista tem mais músicas na base? (`value_counts()` na coluna certa)
- As músicas ficaram mais curtas ao longo do tempo?
- Existe gênero em que dançabilidade e popularidade andem juntas, mesmo
  que no geral não andem?
- Refaça o boxplot do gênero com `playlist_subgenre` em vez de
  `playlist_genre`. O que muda?

Este notebook fica com você. Ele roda de novo a qualquer momento, do
zero, em qualquer máquina com navegador.